# Lab 06 — Build a Synthetic Wordle Curriculum

**Goal:** turn the symbolic expert from Lab 05 into a leakage-safe supervised dataset for Qwen3-0.6B.

This is not a "generate a giant JSON file" lab.

Lab 04 gave us a very specific failure:

```text
post-first-turn history consistency: 0 / 95 = 0%
```

So our dataset should teach the missing capability deliberately.

We will generate three supervision families:

1. **NEXT_GUESS** — imitate the expert action from a real game state.
2. **VALID_CANDIDATE** — decide whether a proposed word is consistent with prior feedback.
3. **CHOOSE_VALID** — choose the consistent word from one valid and one invalid candidate.

The latter two are intentionally easier than optimal action selection. They create a curriculum:

```text
understand feedback
      ↓
recognize valid state transitions
      ↓
choose expert actions
```

By the end of the lab you will have reproducible train/dev/test JSONL files plus dataset statistics ready for Lab 07 full SFT.

## 6.1 Load the expert artifacts

This lab assumes Lab 05 produced:

```text
../data/wordle-answers-original.txt
../data/wordle-patterns-original-2315.npy
../src/tiny_wordle/expert.py
```

We reuse them rather than regenerating Wordle semantics.

In [1]:
from __future__ import annotations

from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
import hashlib
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tiny_wordle.game import Turn, score_string
from tiny_wordle.expert import EntropyExpert

DATA_DIR = Path("../data")
GENERATED_DIR = DATA_DIR / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

ANSWERS = [
    line.strip().upper()
    for line in (DATA_DIR / "wordle-answers-original.txt").read_text().splitlines()
    if line.strip()
]

PATTERNS = np.load(DATA_DIR / "wordle-patterns-original-2315.npy")

assert len(ANSWERS) == 2315
assert PATTERNS.shape == (2315, 2315)

expert = EntropyExpert(ANSWERS, PATTERNS)

WORD_TO_INDEX = expert.word_to_index
ALL_INDICES = expert.all_indices

print("answers:", len(ANSWERS))
print("pattern matrix:", PATTERNS.shape, PATTERNS.dtype)

answers: 2315
pattern matrix: (2315, 2315) uint8


## 6.2 Freeze held-out answers **before** generating examples

This is the most important dataset-engineering decision in the lab.

If we generate all states first and then randomly split rows, states from the same hidden answer can land in both training and validation.

That leaks game-specific information across splits.

Instead:

```text
answer
  ↓
assign split
  ↓
generate all examples for that answer into that split
```

The original Lab 04 words become a fixed test set. `PLANT` was used while tuning the benchmark prompt, so we treat it as a dev/smoke answer rather than a pristine test answer.

In [2]:
DEV_FIXED = {
    "PLANT",
}

TEST_FIXED = {
    "SHORE", "MIGHT", "BRICK", "GHOST", "KNIFE",
    "DOUBT", "FLING", "ROUND", "CHAMP", "WASTE",
    "BLIND", "POINT", "SLATE", "CRANE", "APPLE",
    "SHEEP", "BANAL", "ALLEY", "AUDIO",
}

assert DEV_FIXED.isdisjoint(TEST_FIXED)
assert DEV_FIXED <= set(ANSWERS)
assert TEST_FIXED <= set(ANSWERS)

def stable_bucket(word: str) -> int:
    digest = hashlib.sha256(word.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % 1000

def answer_split(answer: str) -> str:
    if answer in TEST_FIXED:
        return "test"

    if answer in DEV_FIXED:
        return "dev"

    # Roughly 10% of the remaining answer vocabulary goes to dev.
    return "dev" if stable_bucket(answer) < 100 else "train"

split_counts = Counter(answer_split(word) for word in ANSWERS)
split_counts

Counter({'train': 2065, 'dev': 231, 'test': 19})

The fixed test set is deliberately small because it exists to preserve continuity with Lab 04.

Lab 07 can evaluate both:

- this fixed 19-answer benchmark;
- the much larger answer-level dev split.

Do **not** tune hyperparameters on the test answers.

## 6.3 Recreate expert trajectories

We need the state *before* every expert action.

Each record contains:

- hidden answer — metadata only, never shown to Qwen;
- turn number;
- prior history;
- exact surviving candidate set;
- expert next guess;
- entropy;
- whether the full candidate-only expert eventually solves the game.

The hidden answer stays in dataset metadata so we can audit splits and failures.

In [3]:
@dataclass
class ExpertState:
    answer: str
    split: str
    turn: int
    history: list[Turn]
    candidate_indices: list[int]
    expert_guess: str
    expert_entropy: float

def play_expert_states(answer: str, max_turns: int = 6):
    answer = answer.upper()
    candidate_indices = ALL_INDICES.copy()
    history: list[Turn] = []
    states: list[ExpertState] = []

    solved = False

    for turn in range(1, max_turns + 1):
        guess_idx = expert.choose(candidate_indices)
        guess = ANSWERS[guess_idx]
        entropy = expert.entropy(guess_idx, candidate_indices)

        states.append(
            ExpertState(
                answer=answer,
                split=answer_split(answer),
                turn=turn,
                history=list(history),
                candidate_indices=candidate_indices.tolist(),
                expert_guess=guess,
                expert_entropy=entropy,
            )
        )

        feedback = score_string(answer, guess)
        history.append(Turn(guess=guess, feedback=feedback))

        if feedback == "GGGGG":
            solved = True
            break

        candidate_indices = expert.update(
            candidate_indices,
            guess_idx,
            feedback,
        )

        if len(candidate_indices) == 0:
            raise RuntimeError(
                f"expert candidate set became empty for {answer}"
            )

    return states, solved

states, solved = play_expert_states("PLANT")

print("solved:", solved)
for state in states:
    print(
        state.turn,
        state.expert_guess,
        "candidates=", len(state.candidate_indices),
        "history=", [(t.guess, t.feedback) for t in state.history],
    )

solved: True
1 RAISE candidates= 2315 history= []
2 FLOAT candidates= 92 history= [('RAISE', 'BYBBB')]
3 PLANT candidates= 1 history= [('RAISE', 'BYBBB'), ('FLOAT', 'BGBYG')]


## 6.4 Generate all expert states

We keep states from failed trajectories for **state-validity supervision**, because the states themselves are exact.

But for `NEXT_GUESS` imitation, we will initially use only trajectories that eventually solve within six turns.

That avoids teaching Qwen the known candidate-only endgame pathology from Lab 05.

In [4]:
all_states: list[ExpertState] = []
solved_by_answer = {}

for i, answer in enumerate(ANSWERS, 1):
    states, solved = play_expert_states(answer)
    all_states.extend(states)
    solved_by_answer[answer] = solved

print("expert states:", len(all_states))
print("solved answers:", sum(solved_by_answer.values()))
print("failed answers:", sum(not x for x in solved_by_answer.values()))

expert states: 8319
solved answers: 2304
failed answers: 11


## 6.5 Define the model-facing history representation

Keep the representation consistent with the Lab 04 baseline:

```text
R A I S E -> B Y B B B
```

The dataset generator owns presentation. The Wordle environment remains representation-agnostic.

In [5]:
def format_history(history: list[Turn]) -> str:
    if not history:
        return "No previous guesses."

    return "\n".join(
        f"{' '.join(turn.guess)} -> {' '.join(turn.feedback)}"
        for turn in history
    )

print(format_history(states[-1].history))

R A I S E -> B Y B B B
F L O A T -> B Y Y G B
L O C A L -> B G B G G
M O D A L -> B G B G G


## 6.6 Task 1 — NEXT_GUESS

This is straightforward behavioral cloning.

Input:

```text
Task: NEXT_GUESS
History:
R A I S E -> B Y B B B

Return exactly one uppercase five-letter word.
```

Target:

```text
FLOAT
```

We exclude actions from expert trajectories that eventually fail.

In [6]:
def make_next_guess_example(state: ExpertState) -> dict | None:
    if not solved_by_answer[state.answer]:
        return None

    prompt = (
        "Task: NEXT_GUESS\n"
        "You are playing Wordle.\n"
        "Use the game history to choose the next guess.\n"
        "Return exactly one uppercase five-letter word.\n\n"
        "History:\n"
        f"{format_history(state.history)}"
    )

    return {
        "task": "NEXT_GUESS",
        "split": state.split,
        "answer": state.answer,
        "turn": state.turn,
        "candidate_count": len(state.candidate_indices),
        "prompt": prompt,
        "response": state.expert_guess,
    }

example = make_next_guess_example(states[-1])
example

{'task': 'NEXT_GUESS',
 'split': 'train',
 'answer': 'ZONAL',
 'turn': 5,
 'candidate_count': 1,
 'prompt': 'Task: NEXT_GUESS\nYou are playing Wordle.\nUse the game history to choose the next guess.\nReturn exactly one uppercase five-letter word.\n\nHistory:\nR A I S E -> B Y B B B\nF L O A T -> B Y Y G B\nL O C A L -> B G B G G\nM O D A L -> B G B G G',
 'response': 'ZONAL'}

### A critical caveat

Turn 1 has no game state.

Every answer generates essentially the same opening-policy example:

```text
No previous guesses. → RAISE
```

If we keep thousands of duplicates, the dataset will massively over-weight the opener.

So for `NEXT_GUESS` we keep only **one opening example per split**.

## 6.7 Task 2 — VALID_CANDIDATE

The baseline's failure is not merely poor entropy strategy. It cannot reliably carry feedback constraints forward.

So we explicitly supervise:

> Given this history, is this candidate still possible?

Positive examples come from the exact surviving candidate set.

Negative examples are sampled from words outside that set.

This is a much denser signal about Wordle semantics than only showing the expert's chosen action.

In [7]:
RNG = random.Random(42)

def sample_positive_negative(state: ExpertState):
    valid = state.candidate_indices
    valid_set = set(valid)

    positive_idx = RNG.choice(valid)

    while True:
        negative_idx = RNG.randrange(len(ANSWERS))
        if negative_idx not in valid_set:
            break

    return ANSWERS[positive_idx], ANSWERS[negative_idx]

def make_validity_example(
    state: ExpertState,
    candidate: str,
    is_valid: bool,
) -> dict:
    prompt = (
        "Task: VALID_CANDIDATE\n"
        "You are playing Wordle.\n"
        "Given the game history, decide whether the candidate "
        "could still be the hidden answer.\n"
        "Return exactly VALID or INVALID.\n\n"
        "History:\n"
        f"{format_history(state.history)}\n\n"
        f"Candidate: {' '.join(candidate)}"
    )

    return {
        "task": "VALID_CANDIDATE",
        "split": state.split,
        "answer": state.answer,
        "turn": state.turn,
        "candidate_count": len(state.candidate_indices),
        "candidate": candidate,
        "prompt": prompt,
        "response": "VALID" if is_valid else "INVALID",
    }

state = states[-1]
positive, negative = sample_positive_negative(state)

print(make_validity_example(state, positive, True)["prompt"])
print("TARGET:", "VALID")
print()
print("negative:", negative)

Task: VALID_CANDIDATE
You are playing Wordle.
Given the game history, decide whether the candidate could still be the hidden answer.
Return exactly VALID or INVALID.

History:
R A I S E -> B Y B B B
F L O A T -> B Y Y G B
L O C A L -> B G B G G
M O D A L -> B G B G G

Candidate: Z O N A L
TARGET: VALID

negative: ARDOR


We skip turn 1 for validity examples because with no feedback every answer is valid. That teaches almost nothing.

## 6.8 Task 3 — CHOOSE_VALID

Binary classification is useful, but generation-time Wordle requires choosing an action.

So we also create a simple contrastive task:

```text
Option A: CRAKE
Option B: PLANT

Which option is consistent?
```

The output is the **word**, not `A/B`, so Qwen practices generating five-letter actions.

In [8]:
def make_choose_valid_example(state: ExpertState) -> dict | None:
    if state.turn == 1:
        return None

    positive, negative = sample_positive_negative(state)

    # Deterministically alternate order from answer+turn hash.
    positive_first = stable_bucket(state.answer + str(state.turn)) % 2 == 0

    if positive_first:
        option_a, option_b = positive, negative
    else:
        option_a, option_b = negative, positive

    prompt = (
        "Task: CHOOSE_VALID\n"
        "You are playing Wordle.\n"
        "Exactly one option is consistent with all previous feedback.\n"
        "Return exactly the valid five-letter word.\n\n"
        "History:\n"
        f"{format_history(state.history)}\n\n"
        f"Option A: {' '.join(option_a)}\n"
        f"Option B: {' '.join(option_b)}"
    )

    return {
        "task": "CHOOSE_VALID",
        "split": state.split,
        "answer": state.answer,
        "turn": state.turn,
        "candidate_count": len(state.candidate_indices),
        "prompt": prompt,
        "response": positive,
    }

choose_example = make_choose_valid_example(states[-1])
print(choose_example["prompt"])
print("\nTARGET:", choose_example["response"])

Task: CHOOSE_VALID
You are playing Wordle.
Exactly one option is consistent with all previous feedback.
Return exactly the valid five-letter word.

History:
R A I S E -> B Y B B B
F L O A T -> B Y Y G B
L O C A L -> B G B G G
M O D A L -> B G B G G

Option A: Z O N A L
Option B: H U M P H

TARGET: ZONAL


## 6.9 Build the raw curriculum

For each expert state:

- `NEXT_GUESS`: one example, but only from solved expert trajectories;
- `VALID_CANDIDATE`: one positive + one negative after turn 1;
- `CHOOSE_VALID`: one contrastive example after turn 1.

Then deduplicate exact `(prompt, response)` pairs.

This gives state-understanding tasks more representation than turn-1 imitation.

In [14]:
raw_examples = []

opening_kept = set()

for state in all_states:
    next_example = make_next_guess_example(state)

    if next_example is not None:
        if state.turn == 1:
            if state.split not in opening_kept:
                raw_examples.append(next_example)
                opening_kept.add(state.split)
        else:
            raw_examples.append(next_example)

    if state.turn > 1:
        positive, negative = sample_positive_negative(state)

        raw_examples.append(
            make_validity_example(state, positive, True)
        )
        raw_examples.append(
            make_validity_example(state, negative, False)
        )

        choose = make_choose_valid_example(state)
        if choose is not None:
            raw_examples.append(choose)

print("raw examples:", len(raw_examples))

dedup = {}
for ex in raw_examples:
    key = (ex["split"], ex["prompt"], ex["response"])
    dedup[key] = ex

examples = list(dedup.values())

print("deduplicated examples:", len(examples))

raw examples: 23964
deduplicated examples: 19369


## 6.10 Inspect the dataset before trusting it

Synthetic data is seductive because it is cheap.

Cheap garbage is still garbage.

Inspect task counts, split counts, turn distribution, candidate-count distribution, and random samples.

In [16]:
df = pd.DataFrame(examples)

print("TASK COUNTS")
print(df["task"].value_counts())

print("\nSPLIT COUNTS")
print(df["split"].value_counts())

print("\nTASK × SPLIT")
print(pd.crosstab(df["task"], df["split"]))

TASK COUNTS
task
VALID_CANDIDATE    10782
CHOOSE_VALID        6004
NEXT_GUESS          2583
Name: count, dtype: int64

SPLIT COUNTS
split
train    16980
dev       2199
test       190
Name: count, dtype: int64

TASK × SPLIT
split             dev  test  train
task                              
CHOOSE_VALID      602    49   5353
NEXT_GUESS        408    43   2132
VALID_CANDIDATE  1189    98   9495


In [11]:
print("TURN DISTRIBUTION")
print(
    pd.crosstab(
        df["turn"],
        df["task"],
    )
)

TURN DISTRIBUTION
task  CHOOSE_VALID  NEXT_GUESS  VALID_CANDIDATE
turn                                           
1                0           1                0
2             2314         131             3775
3             2183         999             3869
4             1184         919             2247
5              265         207              503
6               58          47              112


In [12]:
df.groupby("task")["candidate_count"].describe()

,count,mean,std,min,25%,50%,75%,max
task,,,,,,,,
CHOOSE_VALID,6004.0,25.595603,40.780308,1.0,1.0,4.0,26.0,168.0
NEXT_GUESS,2304.0,3.610677,48.792914,1.0,1.0,1.0,2.0,2315.0
VALID_CANDIDATE,10506.0,23.824862,39.779887,1.0,1.0,4.0,25.0,168.0


In [13]:
sample = df.sample(6, random_state=42)

for _, row in sample.iterrows():
    print("=" * 80)
    print("TASK:", row["task"])
    print("SPLIT:", row["split"])
    print("ANSWER METADATA:", row["answer"])
    print(row["prompt"])
    print("\nTARGET:", row["response"])

TASK: NEXT_GUESS
SPLIT: train
ANSWER METADATA: TRIED
Task: NEXT_GUESS
You are playing Wordle.
Use the game history to choose the next guess.
Return exactly one uppercase five-letter word.

History:
R A I S E -> Y B G B Y
F R I E D -> B G G G G
C R I E D -> B G G G G
D R I E D -> B G G G G

TARGET: PRIED
TASK: NEXT_GUESS
SPLIT: train
ANSWER METADATA: EVERY
Task: NEXT_GUESS
You are playing Wordle.
Use the game history to choose the next guess.
Return exactly one uppercase five-letter word.

History:
R A I S E -> Y B B B Y
D E T E R -> B Y B Y Y

TARGET: EVERY
TASK: CHOOSE_VALID
SPLIT: train
ANSWER METADATA: GRAIL
Task: CHOOSE_VALID
You are playing Wordle.
Exactly one option is consistent with all previous feedback.
Return exactly the valid five-letter word.

History:
R A I S E -> Y Y Y B B

Option A: T R A I L
Option B: S P L A T

TARGET: TRAIL
TASK: VALID_CANDIDATE
SPLIT: train
ANSWER METADATA: LEMON
Task: VALID_CANDIDATE
You are playing Wordle.
Given the game history, decide whether th

### Look for these failure modes

- hidden answer accidentally appearing in the prompt as metadata;
- malformed five-letter outputs;
- candidate-validity class imbalance;
- thousands of duplicate opening examples;
- train/test answer overlap;
- suspiciously trivial examples dominating the dataset;
- task wording that differs from how Lab 07 will train/infer.

Do not proceed just because `len(dataset)` is large.

## 6.11 Assert split integrity

No hidden answer may appear in more than one split.

This is stronger and easier to reason about than random row splitting.

In [ ]:
answer_splits = (
    df[["answer", "split"]]
    .drop_duplicates()
    .groupby("answer")["split"]
    .nunique()
)

assert answer_splits.max() == 1

train_answers = set(df.loc[df["split"] == "train", "answer"])
dev_answers = set(df.loc[df["split"] == "dev", "answer"])
test_answers = set(df.loc[df["split"] == "test", "answer"])

assert train_answers.isdisjoint(dev_answers)
assert train_answers.isdisjoint(test_answers)
assert dev_answers.isdisjoint(test_answers)

assert TEST_FIXED <= test_answers

print("train answers:", len(train_answers))
print("dev answers:", len(dev_answers))
print("test answers:", len(test_answers))
print("answer-level split integrity passed.")

## 6.12 Validate every target mechanically

The point of symbolic synthetic data is that we can test labels exactly.

For validity tasks:

```text
VALID   → candidate index must be in candidate set
INVALID → candidate index must not be in candidate set
```

For next-guess tasks:

```text
response must equal stored expert action
```

We validate against the source states rather than trusting string generation.

In [ ]:
state_lookup = {
    (s.answer, s.turn): s
    for s in all_states
}

checked = 0

for ex in examples:
    state = state_lookup[(ex["answer"], ex["turn"])]

    if ex["task"] == "VALID_CANDIDATE":
        idx = WORD_TO_INDEX[ex["candidate"]]
        actually_valid = idx in set(state.candidate_indices)
        expected = "VALID" if actually_valid else "INVALID"
        assert ex["response"] == expected

    elif ex["task"] == "NEXT_GUESS":
        assert ex["response"] == state.expert_guess
        assert len(ex["response"]) == 5

    elif ex["task"] == "CHOOSE_VALID":
        idx = WORD_TO_INDEX[ex["response"]]
        assert idx in set(state.candidate_indices)

    checked += 1

print("mechanically validated examples:", checked)

## 6.13 Check for prompt leakage of the hidden answer

The `answer` field is metadata for auditing and splitting. It is **not** part of the model prompt.

There are legitimate cases where the answer word can appear in history or as an option through normal game mechanics, so a raw string search cannot prove leakage.

But we can at least verify that the dataset formatter never emits an explicit `Answer:` field.

In [ ]:
bad = [
    ex for ex in examples
    if "Hidden answer:" in ex["prompt"]
    or "\nAnswer:" in ex["prompt"]
]

assert not bad

print("No explicit hidden-answer field appears in model prompts.")

## 6.14 Token-length inspection with Qwen's tokenizer

Before training, inspect how large these examples actually are.

Long sequences increase activation memory and training cost.

We do not need the model weights here, only the tokenizer.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def formatted_training_text(prompt: str, response: str) -> str:
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

token_lengths = []

for ex in examples:
    text = formatted_training_text(
        ex["prompt"],
        ex["response"],
    )

    token_lengths.append(
        len(tokenizer.encode(text, add_special_tokens=False))
    )

df["token_length"] = token_lengths

df["token_length"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["token_length"], bins=30)
plt.xlabel("Tokens per training example")
plt.ylabel("Examples")
plt.title("Synthetic curriculum sequence lengths")
plt.show()

If the 99th percentile is comfortably small, we can choose a fixed maximum sequence length in Lab 07 without wasting much padding or truncating important examples.

## 6.15 Inspect curriculum difficulty

Candidate count gives us a rough difficulty signal.

A state with one surviving answer is much easier than a state with hundreds.

Let's bin examples by candidate count.

In [ ]:
def difficulty(candidate_count: int) -> str:
    if candidate_count <= 2:
        return "1-2"
    if candidate_count <= 10:
        return "3-10"
    if candidate_count <= 50:
        return "11-50"
    if candidate_count <= 200:
        return "51-200"
    return "201+"

df["difficulty"] = df["candidate_count"].map(difficulty)

pd.crosstab(df["difficulty"], df["task"])

This metadata gives us options in Lab 07:

- train on the whole distribution;
- start with easier states then add harder ones;
- balance by task/difficulty;
- measure performance by difficulty instead of only aggregate loss.

We will begin with the simplest reasonable training distribution and only add curriculum scheduling if evidence says we need it.

## 6.16 Save JSONL splits

JSON Lines keeps the raw dataset transparent and easy to inspect.

Hugging Face `datasets` can load JSON/JSONL directly, so we do not need a proprietary serialization format for this stage.

In [ ]:
OUTPUT_COLUMNS = [
    "task",
    "split",
    "answer",
    "turn",
    "candidate_count",
    "prompt",
    "response",
]

paths = {}

for split in ["train", "dev", "test"]:
    split_df = df.loc[
        df["split"] == split,
        OUTPUT_COLUMNS,
    ].copy()

    path = GENERATED_DIR / f"wordle-sft-{split}.jsonl"

    split_df.to_json(
        path,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    paths[split] = path

    print(
        split,
        len(split_df),
        "->",
        path,
        f"({path.stat().st_size / 1024:.1f} KiB)",
    )

## 6.17 Load the saved files through Hugging Face Datasets

We save plain JSONL, then verify the exact files Lab 07 will consume.

This is a useful boundary: generator code ends here; training code begins from persisted artifacts.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": str(paths["train"]),
        "validation": str(paths["dev"]),
        "test": str(paths["test"]),
    },
)

dataset

In [ ]:
for split in dataset:
    print(split, dataset[split].num_rows)

print("\ncolumns:", dataset["train"].column_names)
print("\nfirst train row:")
dataset["train"][0]

## 6.18 Write a dataset manifest

The manifest captures the decisions that otherwise disappear into notebook state.

Lab 07 should read this before training.

In [ ]:
manifest = {
    "version": 1,
    "generator": "Lab 06 synthetic Wordle curriculum",
    "answer_lexicon": "original Wordle 2315 answers",
    "expert": "candidate-only maximum Shannon entropy",
    "expert_solve_rate_lab05": 2304 / 2315,
    "representation": "space-separated letters and B/Y/G feedback",
    "tasks": [
        "NEXT_GUESS",
        "VALID_CANDIDATE",
        "CHOOSE_VALID",
    ],
    "next_guess_policy": (
        "exclude all action examples from expert trajectories "
        "that fail within six turns"
    ),
    "opening_policy": "one NEXT_GUESS turn-1 example per split",
    "split_policy": (
        "answer-level split before example generation; "
        "Lab04 answers fixed to test; PLANT fixed to dev"
    ),
    "fixed_dev_answers": sorted(DEV_FIXED),
    "fixed_test_answers": sorted(TEST_FIXED),
    "counts": {
        split: int((df["split"] == split).sum())
        for split in ["train", "dev", "test"]
    },
    "task_counts": {
        key: int(value)
        for key, value in df["task"].value_counts().items()
    },
}

manifest_path = GENERATED_DIR / "wordle-sft-manifest.json"

manifest_path.write_text(
    json.dumps(manifest, indent=2)
)

print(manifest_path)
print(json.dumps(manifest, indent=2))

## 6.19 A baseline dataset is not automatically the best dataset

We now have a defensible first curriculum.

We do **not** know whether mixing the three tasks is optimal.

That is an experiment for after the first SFT run.

Useful later ablations include:

```text
NEXT_GUESS only
vs
NEXT_GUESS + VALID_CANDIDATE
vs
all three tasks
```

and:

```text
normal word form
vs
space-separated letters
```

and:

```text
uniform task sampling
vs
difficulty-balanced sampling
```

Do not optimize those before we have one complete end-to-end SFT result.

## 6.20 What will Lab 07 actually train on?

Each row becomes a normal Qwen chat:

```text
USER:
Task: VALID_CANDIDATE
...
Candidate: C R A K E

ASSISTANT:
INVALID
```

or:

```text
USER:
Task: NEXT_GUESS
...
History:
R A I S E -> B Y B B B

ASSISTANT:
FLOAT
```

As in Lab 02, Qwen sees the entire conversation but the supervised loss should apply only to the assistant response tokens.

Lab 02 taught that mechanism on 16 examples.

Lab 07 will apply the same mechanism to this real synthetic curriculum.

# Lab 06 checkpoint

You should now be able to explain:

1. Why we split by hidden answer before generating examples.
2. Why random row-level train/test splitting would leak related states.
3. Why `NEXT_GUESS` alone may provide weak supervision for the 0% history-consistency failure.
4. Why `VALID_CANDIDATE` and `CHOOSE_VALID` directly train feedback semantics.
5. Why failed expert trajectories can still contain valid state supervision.
6. Why we exclude failed trajectories from initial expert-action imitation.
7. Why thousands of duplicate opening examples would distort the dataset.
8. Why synthetic labels should be mechanically validated.
9. Why raw JSONL plus a manifest is a useful training-data boundary.
10. Why dataset size alone says almost nothing about dataset quality.

## Send me these outputs

Work through the notebook and send:

- answer-level split counts;
- `expert states`, solved answers, failed answers;
- raw and deduplicated example counts;
- task counts;
- split counts;
- answer-level split integrity result;
- mechanically validated example count;
- token-length statistics;
- saved JSONL row counts;
- Hugging Face Dataset split sizes.

Then we will design **Lab 07 — Full SFT** around the dataset we actually produced rather than guessing training hyperparameters in advance.